### 2 - 多臂老虎机


#### 伯努利多臂老虎机环境 Environment

In [1]:
# 导入需要使用的库,其中numpy是支持数组和矩阵运算的科学计算库,而matplotlib是绘图库
import numpy as np
import matplotlib.pyplot as plt


class BernoulliBandit:
    """ 伯努利多臂老虎机,输入K表示拉杆个数 """
    def __init__(self, K):
        self.probs = np.random.uniform(size=K)  # 随机生成K个0～1的数,作为拉动每根拉杆的获奖
        # 概率
        self.best_idx = np.argmax(self.probs)  # 获奖概率最大的拉杆
        self.best_prob = self.probs[self.best_idx]  # 最大的获奖概率
        self.K = K

    def step(self, k):
        # 当玩家选择了k号拉杆后,根据拉动该老虎机的k号拉杆获得奖励的概率返回1（获奖）或0（未
        # 获奖）
        if np.random.rand() < self.probs[k]:
            return 1
        else:
            return 0


np.random.seed(1)  # 设定随机种子,使实验具有可重复性
K = 10
bandit_10_arm = BernoulliBandit(K)
print("随机生成了一个%d臂伯努利老虎机" % K)
print("获奖概率最大的拉杆为%d号,其获奖概率为%.4f" %
      (bandit_10_arm.best_idx, bandit_10_arm.best_prob))

# 随机生成了一个10臂伯努利老虎机
# 获奖概率最大的拉杆为1号,其获奖概率为0.7203

随机生成了一个10臂伯努利老虎机
获奖概率最大的拉杆为1号,其获奖概率为0.7203


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. 多臂老虎机环境 Environment
# ============================================================

class BernoulliBandit:
    """伯努利多臂老虎机, K 表示拉杆个数"""

    def __init__(self, K):
        self.probs = np.random.uniform(size=K)      # 每根拉杆真实中奖概率
        self.best_idx = np.argmax(self.probs)       # 最优拉杆编号
        self.best_prob = self.probs[self.best_idx]  # 最优拉杆中奖概率
        self.K = K                                  # 拉杆数量

    def step(self, k):
        """拉第 k 根杆一次，返回奖励 0 或 1"""
        if np.random.rand() < self.probs[k]:
            return 1
        else:
            return 0


# ============================================================
# 2. 多臂老虎机算法通用框架 Solver
# ============================================================

class Solver:
    """多臂老虎机算法基本框架"""

    def __init__(self, bandit):
        self.bandit = bandit
        self.counts = np.zeros(self.bandit.K)  # 每根拉杆被选择的次数
        self.regret = 0.                       # 当前累计懊悔
        self.actions = []                      # 每一步选择的动作
        self.regrets = []                      # 每一步的累计懊悔

    def update_regret(self, k):
        """更新累计懊悔"""
        self.regret += self.bandit.best_prob - self.bandit.probs[k]
        self.regrets.append(self.regret)

    def run_one_step(self):
        """具体策略负责实现：这一轮选哪根拉杆"""
        raise NotImplementedError

    def run(self, num_steps):
        """运行算法 num_steps 次"""
        for _ in range(num_steps):

            # 具体策略决定选择哪根拉杆
            k = self.run_one_step()

            # 记录该拉杆被选择了一次
            self.counts[k] += 1

            # 记录动作
            self.actions.append(k)

            # 更新累计懊悔
            self.update_regret(k)


# ============================================================
# 3. ε-Greedy 策略
# ============================================================

class EpsilonGreedy(Solver):
    """ε-贪心算法，继承 Solver"""

    def __init__(self, bandit, epsilon=0.01, init_prob=1.0):

        # 先初始化 Solver 中的公共变量
        super(EpsilonGreedy, self).__init__(bandit)

        self.epsilon = epsilon

        # 初始化每根拉杆的期望奖励估计
        self.estimates = np.array(
            [init_prob] * self.bandit.K
        )

    def run_one_step(self):

        # ε 概率：随机探索
        if np.random.random() < self.epsilon:
            k = np.random.randint(0, self.bandit.K)

        # 1-ε 概率：选择当前估计奖励最大的拉杆
        else:
            k = np.argmax(self.estimates)

        # 真正拉第 k 根杆，与环境交互得到奖励
        r = self.bandit.step(k)

        # 用当前奖励增量式更新第 k 根杆的期望奖励估计
        self.estimates[k] += (
            1. / (self.counts[k] + 1)
            * (r - self.estimates[k])
        )

        # 返回本轮选择的拉杆编号
        return k


# ============================================================
# 4. 创建环境 + 创建策略 + 运行算法
# ============================================================

np.random.seed(1)

K = 10

# 创建一个 10 臂老虎机环境
bandit_10_arm = BernoulliBandit(K)

print("随机生成了一个%d臂伯努利老虎机" % K)
print(
    "获奖概率最大的拉杆为%d号，其获奖概率为%.4f"
    % (bandit_10_arm.best_idx, bandit_10_arm.best_prob)
)


# 创建 ε-Greedy 算法
epsilon_greedy_solver = EpsilonGreedy(
    bandit=bandit_10_arm,
    epsilon=0.01
)


# 运行 5000 次
epsilon_greedy_solver.run(5000)


# ============================================================
# 5. 输出结果
# ============================================================

print("各拉杆被选择次数：")
print(epsilon_greedy_solver.counts)

print("各拉杆最终期望奖励估计：")
print(epsilon_greedy_solver.estimates)

print("最终累计懊悔：")
print(epsilon_greedy_solver.regret)


# ============================================================
# 6. 画累计懊悔曲线
# ============================================================

plt.plot(epsilon_greedy_solver.regrets)

plt.xlabel("Steps")
plt.ylabel("Cumulative Regret")
plt.title("Epsilon-Greedy")

plt.show()